# Crop Classification Exploration: Part 1 - API Data Access

This notebook demonstrates direct API-based access to Sentinel-2 and USDA Cropland Data Layer (CDL) data for two study areas (California and Arkansas) using Microsoft Planetary Computer. 

**Objective:**
1. Access multi-source satellite data without downloading large raw files.
2. Visualize NDVI time-series for phenology analysis.
3. Study the distribution of crop classes in the selected regions.

In [ ]:
import os
import sys
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import pandas as pd
import pystac_client
import planetary_computer
import stackstac

# Add root directory to path to import config
sys.path.append(os.path.abspath('..'))
import config

## 1. Helper Functions for STAC Access

In [ ]:
def get_stac_data(area_name, collection, bbox, datetime):
    """Fetch STAC items and return a signed collection."""
    catalog = pystac_client.Client.open(
        config.STAC_API_URL,
        modifier=planetary_computer.sign_inplace,
    )

    search = catalog.search(
        collections=[collection],
        bbox=bbox,
        datetime=datetime,
    )
    
    items = search.item_collection()
    print(f"[{area_name}] Found {len(items)} items for {collection}")
    return items

## 2. Exploration Analysis (California & Arkansas)

In [ ]:
def analyze_area(area_name, epsg):
    print(f"\n--- Exploratory Analysis for {area_name} ---")
    bbox = config.STUDY_AREAS[area_name]["bbox"]
    
    # 2.1 Fetch and Filter Sentinel-2
    s2_items = get_stac_data(area_name, config.S2_COLLECTION, bbox, config.DATE_RANGE)
    s2_items = [i for i in s2_items if i.properties["eo:cloud_cover"] < config.CLOUD_PERCENT]
    print(f"[{area_name}] {len(s2_items)} items after cloud filtering (<{config.CLOUD_PERCENT}%)")

    # 2.2 Fetch CDL
    cdl_items = get_stac_data(area_name, config.CDL_COLLECTION, bbox, "2020-01-01/2020-12-31")

    # 2.3 Create DataCubes (Lazy Loading)
    s2_stack = stackstac.stack(s2_items, assets=["B04", "B08"], bounds_latlon=bbox, epsg=epsg, resolution=10)
    cdl_stack = stackstac.stack(cdl_items, assets=["cropland"], bounds_latlon=bbox, epsg=epsg, resolution=10)

    # 2.4 Compute NDVI Time-Series (Sample Point from middle)
    red = s2_stack.sel(band="B04")
    nir = s2_stack.sel(band="B08")
    ndvi = (nir - red) / (nir + red)
    
    # Compute mean NDVI for the patch to see regional phenology
    ndvi_mean = ndvi.mean(dim=["x", "y"]).compute()
    
    plt.figure(figsize=(12, 5))
    ndvi_mean.plot(marker='o', linestyle='-', color='forestgreen')
    plt.title(f"Mean NDVI Time-Series for {area_name} (2020)")
    plt.ylabel("NDVI")
    plt.grid(True)
    plt.show()

    # 2.5 Plot Crop Distribution
    cdl_vals = cdl_stack.isel(time=0, band=0).compute().values.flatten()
    unique, counts = np.unique(cdl_vals[cdl_vals > 0], return_counts=True)
    
    plt.figure(figsize=(10, 5))
    plt.bar(unique.astype(str), counts, color='teal')
    plt.title(f"Crop Class Distribution (CDL ID) for {area_name}")
    plt.xticks(rotation=45)
    plt.show()

# Run for both areas
analyze_area("California", 32611)
analyze_area("Arkansas", 32615)